# 04 — BAR-ATH (only if H1 passed; aggregates only)

If H1 failed, nothing is read or written for BAR-ATH. If it passed: 3 full-data `exp5_touch` fits, exp_3's fusion rule (tactical input if rejected), `exp_5_candidate_provider.csv` (decoder selected on provider labels) and, only if `L_ctrl` non-inferiority holds, `exp_5_candidate_ctrl.csv`; official validator; byte-identical rebuild from 3 independent fits; label-free plausibility. No public-window check.

In [1]:
from pathlib import Path
import os, sys, json, time
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp5_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_5')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp5_common as C
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp5_protocol as P
assert C.public_check_spent(), 'exp_1 public check marker missing'

PREREG verified dc28ebfcacfa2676 | frozen 2026-11-10T09:06:10+00:00


In [2]:
decision = C1.read_json(C.RESULTS / 'decision.json')
if not decision['H1']['passed']:
    summary = {'H1_passed': False, 'files_written': [], 'bar_ath_read': False,
               'ci_clause': decision['H1']['ci_clause'], 'seed_clause': decision['H1']['seed_clause'],
               'statement': decision['adoption']['statement']}
    C1.write_json(C.RESULTS / 'target_none.json', summary)
    print(json.dumps(summary, indent=1))
else:
    import exp5_target as T
    missing_full = sum(not T.model_path(a).exists() for a in T.MEMBERS)
    used = P.proxy_fit_count() + 1 + T.target_fit_count()
    assert used + missing_full + C.PLANNED_FITS['rebuild'] <= C.FIT_CAP, 'fit cap reached'
    t0 = time.time()
    match, item, fusion = T.inputs()
    models = {arm: T.full_model(arm) for arm in T.MEMBERS}
    summary = T.build_all(match, item, models)
    summary['fusion'] = fusion
    summary['reproduction'] = T.reproduce(match, item)
    summary['exp3_files'] = T.exp3_files()
    summary['files_written'] = [summary[ls]['file'] for ls in decision['adoption']['label_sets']]
    summary['fits_used'] = {'proxy': P.proxy_fit_count(), 'reproduction_check': 1, 'full_data': T.target_fit_count(),
                            'rebuild': summary['reproduction']['independent_fits']}
    C1.write_json(C.RESULTS / 'target_candidates.json', summary)
    keys = ('file', 'sha256', 'dead_share', 'possessions_per_min', 'mean_possession_s', 'possession_count', 'decoder')
    for ls in decision['adoption']['label_sets']:
        print(ls, json.dumps({k: summary[ls].get(k) for k in keys}))
    print('input:', fusion['input'], '| reproduction byte-identical:', summary['reproduction']['byte_identical'], f'| {time.time() - t0:.0f}s')
print('exp_5 proxy fits cached:', P.proxy_fit_count(), '| + 1 reproduction check')

provider {"file": "exp_5_candidate_provider.csv", "sha256": "1b9795cb2cf75363518600cc4b6c7c14f1b053613d7a65d19ef5df3ba742da62", "dead_share": 0.48837004349824054, "possessions_per_min": 2.5022504516640756, "mean_possession_s": 12.268075471698115, "possession_count": 265, "decoder": {"temperature": 1.1147855174930903, "lambda_team": 0.5, "lambda_dead": 4.0, "gamma": 0.0}}
input: tactical | reproduction byte-identical: True | 364s
exp_5 proxy fits cached: 84 | + 1 reproduction check


In [3]:
GUARD_FILE = 'write_guard_04.json'
before = C1.read_json(C.RESULTS / 'write_guard_before.json')
now = C.write_manifest()
guard = {name: {'unchanged': now[name] == state, 'files_before': state['files'], 'files_now': now[name]['files']}
         for name, state in before['folders'].items()}
C1.write_json(C.RESULTS / GUARD_FILE, {'checked_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
                                       'since': before['taken_at_utc'], 'folders': guard})
print('nothing written under exp_0-exp_4 / data/ since freezing:', all(g['unchanged'] for g in guard.values()), guard)


nothing written under exp_0-exp_4 / data/ since freezing: True {'exp_0': {'unchanged': True, 'files_before': 42630, 'files_now': 42630}, 'exp_1': {'unchanged': True, 'files_before': 215, 'files_now': 215}, 'exp_2': {'unchanged': True, 'files_before': 312, 'files_now': 312}, 'exp_3': {'unchanged': True, 'files_before': 407, 'files_now': 407}, 'exp_4': {'unchanged': True, 'files_before': 638, 'files_now': 638}, 'data': {'unchanged': True, 'files_before': 12, 'files_now': 12}}
